<a href="https://colab.research.google.com/github/aapaul29/Distance-Recognition-SML1-/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SML Project 1
This notebook works in both **VS Code (local)** and **Google Colab**.

## 1. Environment Detection & Setup

In [131]:
import sys
import os

IN_COLAB = 'google.colab' in sys.modules
print(f"Running in {'Google Colab' if IN_COLAB else 'VS Code / local'}")

Running in VS Code / local


## 2. Google Colab: Mount Drive & Navigate to Project
This cell only runs on Colab. Adjust `COLAB_PROJECT_PATH` to where you uploaded the project folder in your Google Drive.

In [132]:
# ---------------------------------------------------------------
# COLAB ONLY — skip if running locally
# Adjust this path to match your Google Drive folder structure.
COLAB_PROJECT_PATH = "/content/drive/MyDrive/Aaryan/Uni/BS Mech Eng ETH Zurich/2nd year/Semester 4/Stochastics & Machine Learning/Distance-Recognition-SML1-"
# ---------------------------------------------------------------

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd $COLAB_PROJECT_PATH
    print(f"Working directory: {os.getcwd()}")

    # Unzip data if not already extracted
    data_zip = os.path.join(COLAB_PROJECT_PATH, "data.zip")
    data_dir = os.path.join(COLAB_PROJECT_PATH, "data")
    if os.path.exists(data_zip) and not os.path.exists(data_dir):
        print("Unzipping data...")
        os.makedirs(data_dir, exist_ok=True)
        !unzip -q $data_zip -d $data_dir
        print("Done.")
    else:
        print("Data directory already exists or no zip found — skipping unzip.")
else:
    print("Local environment — no mounting needed.")

Local environment — no mounting needed.


## 3. Install / Verify Dependencies

In [133]:
# On Colab, install any missing packages from requirements.txt.
# Locally, this is a no-op if your conda env is already set up.
if IN_COLAB:
    !pip install -q -r requirements.txt

## 4. Auto-reload & Imports

In [134]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from utils import load_config, load_dataset, load_test_dataset, print_results, save_results

# Add further sklearn imports here as needed.
# Note: SVRs are NOT allowed in this project.

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 5. Load Config & Data

In [135]:
config = load_config()

images, distances = load_dataset(config, split="train")
print(f"[INFO]: Dataset loaded with {len(images)} samples.")
print(f"[INFO]: Feature dimensionality: {images.shape[1]}")

[INFO]: Configs are loaded with: 
 {'data_dir': PosixPath('data'), 'load_rgb': False, 'downsample_factor': 3}
[INFO]: Dataset loaded with 3000 samples.
[INFO]: Feature dimensionality: 10000


## 6. Your Implementation
Implement preprocessing, model training, and evaluation below.

In [136]:
# TODO: Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.ndimage import sobel, uniform_filter
import numpy as np

ds = config["downsample_factor"]
H = W = 300 // ds
C = 3 if config["load_rgb"] else 1

def get_grid_stats_features(X, H, W):
    all_feat = []
    for img in X:
        img_2d = img.reshape(H, W).astype(np.float32)
        # Bottom Weighting
        weights = np.linspace(0.2, 1.0, H).reshape(-1, 1)
        img_2d = img_2d * weights

        grid_features = []
        grid_h, grid_w = H // 4, W // 4
        for i in range(4):
            for j in range(4):
                cell = img_2d[i*grid_h:(i+1)*grid_h, j*grid_w:(j+1)*grid_w]
                grid_features.extend([cell.mean(), cell.var()])
        strip_features = []
        strip_h = H // 8
        for i in range(8):
            strip = img_2d[i*strip_h:(i+1)*strip_h, :]
            strip_features.extend([strip.mean(), strip.var()])
        hist, _ = np.histogram(img_2d, bins=16, range=(0, 255))
        hist = hist.astype(np.float32) / hist.sum()
        global_feat = np.array([img_2d.mean(), img_2d.std()])
        col_min = np.min(img_2d, axis=0).astype(np.float32)
        all_feat.append(np.concatenate([grid_features, strip_features, hist, global_feat, col_min]))
    return np.array(all_feat)

def get_edge_features(X, H, W):
    all_feat = []
    for img in X:
        img_2d = img.reshape(H, W).astype(np.float32)
        # Bottom Weighting
        weights = np.linspace(0.2, 1.0, H).reshape(-1, 1)
        img_2d = img_2d * weights

        sx = sobel(img_2d, axis=1)
        sy = sobel(img_2d, axis=0)
        hist_sx, _ = np.histogram(sx, bins=16)
        hist_sy, _ = np.histogram(sy, bins=16)
        hist_sx = hist_sx.astype(np.float32) / (hist_sx.sum() + 1e-8)
        hist_sy = hist_sy.astype(np.float32) / (hist_sy.sum() + 1e-8)
        sobel_feat = np.array([sx.mean(), sx.var(), sy.mean(), sy.var()])
        grad_map = np.abs(np.diff(img_2d, axis=0))
        strong_edges = grad_map > 15
        rows = np.where(strong_edges, np.arange(H-1).reshape(-1, 1), -1)
        max_rows = rows.max(axis=0)
        floor_scan = np.where(max_rows >= 0, max_rows / H, 0).astype(np.float32)
        strip_grad = []
        strip_h = H // 8
        for i in range(8):
            strip = img_2d[i*strip_h:(i+1)*strip_h, :]
            sx_s = sobel(strip, axis=1)
            sy_s = sobel(strip, axis=0)
            mag = np.sqrt(sx_s**2 + sy_s**2)
            strip_grad.extend([mag.mean(), mag.var()])
        all_feat.append(np.concatenate([sobel_feat, hist_sx, hist_sy, floor_scan, strip_grad]))
    return np.array(all_feat)

def augment(X, y, H, W, C):
    X_aug, y_aug = [X], [y]
    X_flip = X.reshape(-1, H, W, C)[:, :, ::-1, :].reshape(len(X), -1)
    X_aug.append(X_flip); y_aug.append(y)
    X_dark = np.clip(X * 0.8, 0, 255)
    X_aug.append(X_dark); y_aug.append(y)
    X_bright = np.clip(X * 1.2, 0, 255)
    X_aug.append(X_bright); y_aug.append(y)
    X_blur = []
    for img in X:
        img_2d = img.reshape(H, W).astype(np.float32)
        blurred = uniform_filter(img_2d, size=3).reshape(-1)
        X_blur.append(blurred)
    X_aug.append(np.array(X_blur)); y_aug.append(y)
    X_erase = []
    for img in X:
        img_2d = img.reshape(H, W).copy()
        x1, y1 = np.random.randint(0, W-10), np.random.randint(0, H-10)
        img_2d[y1:y1+10, x1:x1+10] = 0
        X_erase.append(img_2d.reshape(-1))
    X_aug.append(np.array(X_erase)); y_aug.append(y)
    X_shift = []
    for img in X:
        img_2d = img.reshape(H, W)
        shifted = np.roll(img_2d, 3, axis=0)
        X_shift.append(shifted.reshape(-1))
    X_aug.append(np.array(X_shift)); y_aug.append(y)
    X_shift2 = []
    for img in X:
        img_2d = img.reshape(H, W)
        shifted = np.roll(img_2d, -3, axis=0)
        X_shift2.append(shifted.reshape(-1))
    X_aug.append(np.array(X_shift2)); y_aug.append(y)
    return np.vstack(X_aug), np.concatenate(y_aug)

# Split zuerst
X_train_raw, X_val_raw, y_train_raw, y_val = train_test_split(
    images, distances, test_size=0.15, random_state=42
)

X_aug, y_train = augment(X_train_raw, y_train_raw, H, W, C)

X2_train = get_grid_stats_features(X_aug, H, W)
X2_val = get_grid_stats_features(X_val_raw, H, W)
scaler2 = StandardScaler()
X2_train = scaler2.fit_transform(X2_train)
X2_val = scaler2.transform(X2_val)

X3_train = get_edge_features(X_aug, H, W)
X3_val = get_edge_features(X_val_raw, H, W)
scaler3 = StandardScaler()
X3_train = scaler3.fit_transform(X3_train)
X3_val = scaler3.transform(X3_val)

X_train_all = np.hstack([X2_train, X3_train])
X_val_all = np.hstack([X2_val, X3_val])

print(f"Train: {X_train_all.shape}, Val: {X_val_all.shape}")

Train: (20400, 318), Val: (450, 318)


In [137]:
# TODO: Model definition & training
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor, StackingRegressor
from sklearn.linear_model import Ridge

estimators = [
    ("hgb", HistGradientBoostingRegressor(max_iter=500, max_depth=7, learning_rate=0.05, random_state=42)),
    ("et", ExtraTreesRegressor(n_estimators=200, max_depth=20, n_jobs=-1, random_state=42)),
]

model = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(alpha=10.0),
    cv=5,
    n_jobs=-1
)

print("Training läuft...")
model.fit(X_train_all, y_train)
print("Fertig!")

Training läuft...
Fertig!


In [138]:
# TODO: Evaluation on training / validation set
print("Validation:")
print_results(y_val, model.predict(X_val_all))
print("Training:")
print_results(y_train, model.predict(X_train_all))

Validation:
MAE: 12.412
R2: 82.976
Training:
MAE: 5.55
R2: 97.269


## 7. Predict on Test Set & Save

In [139]:
# Retrain auf vollem Dataset
print("Retrain auf vollem Dataset...")
X_full_raw = np.vstack([X_train_raw, X_val_raw])
y_full_raw = np.concatenate([y_train_raw, y_val])

X_aug_full, y_full = augment(X_full_raw, y_full_raw, H, W, C)

X2_full = get_grid_stats_features(X_aug_full, H, W)
scaler2_full = StandardScaler()
X2_full = scaler2_full.fit_transform(X2_full)

X3_full = get_edge_features(X_aug_full, H, W)
scaler3_full = StandardScaler()
X3_full = scaler3_full.fit_transform(X3_full)

X_full_all = np.hstack([X2_full, X3_full])
model.fit(X_full_all, y_full)
print("Retrain fertig!")

test_images_raw = load_test_dataset(config)
X_test_raw = np.array(test_images_raw)
X2_test = scaler2_full.transform(get_grid_stats_features(X_test_raw, H, W))
X3_test = scaler3_full.transform(get_edge_features(X_test_raw, H, W))
X_test_all = np.hstack([X2_test, X3_test])

test_pred = model.predict(X_test_all)
save_results(test_pred)
print("Gespeichert!")

Retrain auf vollem Dataset...
Retrain fertig!
Gespeichert!
